In [ ]:
import os

KAFKA_HOST = os.environ["KAFKA_HOST"]
KAFKA_PORT = os.environ["KAFKA_PORT"]

KAFKA_BOOTSTRAP_SERVERS = f"{KAFKA_HOST}:{KAFKA_PORT}"

SPARK_CONNECT_HOST = os.environ["SPARK_CONNECT_HOST"]
SPARK_CONNECT_PORT = os.environ["SPARK_CONNECT_PORT"]

SPARK_CONNECT_SERVER = f"{SPARK_CONNECT_HOST}:{SPARK_CONNECT_PORT}"

print(f"KAFKA_BOOTSTRAP_SERVERS: {KAFKA_BOOTSTRAP_SERVERS}")
print(f"SPARK_CONNECT_SERVER: {SPARK_CONNECT_SERVER}")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, LongType, StringType

TOPIC = "dbserver1.public.book_references"  # adjust to your actual Debezium topic

spark = SparkSession.builder \
    .appName("debezium-cdc-append") \
    .remote(f"sc://{SPARK_CONNECT_SERVER}") \
    .getOrCreate()

spark.sql("USE polaris")
spark.sql("USE NAMESPACE COLLADO_TEST.PUBLIC")

# recreate table to hold raw CDC events instead of current-state rows
spark.sql("DROP TABLE IF EXISTS TEST_TABLE_CDC")
spark.sql("""
    CREATE TABLE TEST_TABLE_CDC (
        id bigint,
        data string,
        op string,
        ts_ms long
    )
    USING iceberg
""")

row_schema = StructType([
    StructField("id", LongType(), False),
    StructField("data", StringType(), True),
])

envelope_schema = StructType([
    StructField("before", row_schema, True),
    StructField("after", row_schema, True),
    StructField("op", StringType(), True),
    StructField("ts_ms", LongType(), True),
])

raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load()

parsed = raw.selectExpr("CAST(value AS STRING) as json") \
    .select(from_json(col("json"), envelope_schema).alias("d")) \
    .select(
        col("d.after.id").alias("id"),
        col("d.after.data").alias("data"),
        col("d.op").alias("op"),
        col("d.ts_ms").alias("ts_ms"),
    )

query = parsed.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/checkpoints/test_table_cdc_append") \
    .trigger(processingTime="10 seconds") \
    .toTable("TEST_TABLE_CDC")

query.awaitTermination()